## 1. Imports

In [144]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from collections import defaultdict

# Scikit-learn & Survival Analysis
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

# Load Data
# (Adjust paths if necessary)
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

# Basic Data Preview
print("Clinical Data Shape:", df.shape)
print("Molecular Data Shape:", maf_df.shape)

Clinical Data Shape: (3323, 9)
Molecular Data Shape: (10935, 11)


## 2. Target Cleaning

In [145]:
# Cell 2: Target Cleaning
# Drop rows with missing target values
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Ensure correct types
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Merge target into training data to keep indices aligned
df = df.merge(target_df[['ID', 'OS_YEARS', 'OS_STATUS']], on='ID', how='inner')

## 3. Parsing functions

In [146]:
# Cell 3: Improved Parsing Functions

def parse_PROTEIN_CHANGE(protein):
    """Extracts the amino acid position (hotspot) from protein change string."""
    protein = str(protein)
    if len(protein) == 0 or protein == 'nan':
        return pd.NA
    
    # Regex to find the number in p.R882H -> 882
    match = re.search(r"(\d+)", protein)
    if match:
        return match.group(1)
    return pd.NA

def parse_GENE(gene):
    """Returns the full gene name in uppercase."""
    gene = str(gene)
    if len(gene) == 0 or gene == 'nan':
        return pd.NA
    return gene.upper()

def parse_CYTO(iscn):
    """
    Parses ISCN cytogenetic strings to capture:
    1. Numerical abnormalities (+8, -7)
    2. Structural abnormalities (t(8;21), inv(16)) -> Critical for ELN Risk
    3. Complex Karyotype (>=3 abnormalities)
    """
    iscn = str(iscn).upper().replace(" ", "")
    results = defaultdict(int)
    
    # 1. Detect Complex Karyotype
    # Split by clone '/' and count abnormalities in the most complex clone
    clones = iscn.split("/")
    max_abnormalities = 0
    for clone in clones:
        # Count +, -, del, add, inv, t
        abnormalities = re.findall(r"([+-]\d+|DEL|ADD|INV|T\(|DER)", clone)
        if len(abnormalities) > max_abnormalities:
            max_abnormalities = len(abnormalities)
            
    if max_abnormalities >= 3:
        results["Complex_Karyotype"] = 1

    # 2. Capture Specific Translocations/Inversions (ELN Risk definitions)
    # Matches t(8;21), inv(16), t(15;17), etc.
    structural = re.findall(r"(T|INV)\((\d+|X|Y)[;]?(\d+|X|Y)?\)", iscn)
    for type_, chrom1, chrom2 in structural:
        chrom2_str = f";{chrom2}" if chrom2 else ""
        key = f"{type_}({chrom1}{chrom2_str})"
        results[key] = 1

    # 3. Numeric changes (Gains/Losses)
    # Look for +8, -7, -5, etc not preceded by a number (to avoid chromosome coordinates)
    numeric_changes = re.findall(r"(?<![0-9])([+-])(\d+|X|Y)(?=[,/]|$)", iscn)
    for sign, num in numeric_changes:
        key = f"{sign}{num}"
        results[key] = 1

    if not results:
        results["normal"] = 1
        
    return dict(results)

## 4. Data processing

In [147]:
# Cell 4: Apply Cytogenetics Parsing

# 1. Calculate Nmut (Mutation Count)
if 'Nmut' not in df.columns:
    nmut_train = maf_df.groupby('ID').size().reset_index(name='Nmut')
    nmut_test = maf_eval.groupby('ID').size().reset_index(name='Nmut')
    
    df = df.merge(nmut_train, on='ID', how='left').fillna({'Nmut': 0})
    df_eval = df_eval.merge(nmut_test, on='ID', how='left').fillna({'Nmut': 0})

# 2. Apply Parse CYTO
total_features = ['Nmut']

for dataset in [df, df_eval]:
    # Apply parser
    cyto_dicts = dataset['CYTOGENETICS'].apply(parse_CYTO)
    
    # Convert list of dicts to DataFrame
    cyto_df = pd.DataFrame(cyto_dicts.tolist(), index=dataset.index).fillna(0)
    
    # Join back
    for col in cyto_df.columns:
        if col not in dataset.columns:
            dataset[col] = cyto_df[col]
        
# Add valid cyto columns to feature list
# (We only take columns that appear in the training set)
cyto_cols = [c for c in df.columns if any(x in c for x in ['+', '-', 't(', 'inv(', 'Complex', 'normal'])]
total_features.extend(cyto_cols)

print(f"Added {len(cyto_cols)} cytogenetic features.")

Added 49 cytogenetic features.


/tmp/ipykernel_68591/1299362163.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[col] = cyto_df[col]
/tmp/ipykernel_68591/1299362163.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[col] = cyto_df[col]
/tmp/ipykernel_68591/1299362163.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.

## 5. Molecular features with VAF weighting

In [148]:
# Cell 5: Molecular Features with VAF Weighting

def process_molecular_features():
    global df, df_eval, total_features
    
    # 1. GENE features (Weighted by Max VAF)
    print("Processing Genes with VAF weights...")
    
    # Clean gene names first
    maf_df['GENE_CLEAN'] = maf_df['GENE'].apply(parse_GENE)
    maf_eval['GENE_CLEAN'] = maf_eval['GENE'].apply(parse_GENE)
    
    gene_pivot_train = maf_df.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
    gene_pivot_test = maf_eval.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
    
    # Rename columns
    gene_pivot_train.columns = [f"GENE_{c}" for c in gene_pivot_train.columns]
    gene_pivot_test.columns = [f"GENE_{c}" for c in gene_pivot_test.columns]
    
    # Align columns: Ensure test has exactly the same columns as train, filling 0s where missing
    gene_pivot_test = gene_pivot_test.reindex(columns=gene_pivot_train.columns, fill_value=0)
            
    # Merge into main dataframes
    df = df.merge(gene_pivot_train, on='ID', how='left').fillna(0)
    df_eval = df_eval.merge(gene_pivot_test, on='ID', how='left').fillna(0)
    
    total_features.extend(gene_pivot_train.columns.tolist())
    
    # 2. EFFECT and PROTEIN_CHANGE (One-Hot Encoding)
    for feature in ['EFFECT', 'PROTEIN_CHANGE']:
        print(f"Processing {feature}...")
        
        # Apply parsing
        if feature == 'PROTEIN_CHANGE':
            maf_df['temp_feat'] = maf_df[feature].apply(parse_PROTEIN_CHANGE)
            maf_eval['temp_feat'] = maf_eval[feature].apply(parse_PROTEIN_CHANGE)
        else:
            maf_df['temp_feat'] = maf_df[feature]
            maf_eval['temp_feat'] = maf_eval[feature]
            
        # Get Dummies
        dummies_train = pd.get_dummies(maf_df['temp_feat'], prefix=feature)
        dummies_train['ID'] = maf_df['ID']
        # Aggregate by ID (max -> if present 1, else 0)
        agg_train = dummies_train.groupby('ID').max()
        
        dummies_test = pd.get_dummies(maf_eval['temp_feat'], prefix=feature)
        dummies_test['ID'] = maf_eval['ID']
        agg_test = dummies_test.groupby('ID').max()
        
        # Align columns: Force test to have same structure as train
        # This replaces the loop that caused the warning
        agg_test = agg_test.reindex(columns=agg_train.columns, fill_value=0)
        
        # Identify columns to add (exclude ID if it's in the index/columns weirdly)
        common_cols = list(agg_train.columns)
        
        # Merge
        df = df.merge(agg_train, on='ID', how='left').fillna(0)
        df_eval = df_eval.merge(agg_test, on='ID', how='left').fillna(0)
        
        total_features.extend(common_cols)

process_molecular_features()

Processing Genes with VAF weights...
Processing EFFECT...
Processing PROTEIN_CHANGE...


## 6. Interactions and clinical preparation

In [149]:
# Cell 6: Interactions and Clinical Prep

# 1. Add Biological Interactions
def add_interactions(data):
    # NPM1 mutated + FLT3 Wildtype (Good Prognosis)
    if 'GENE_NPM1' in data.columns and 'GENE_FLT3' in data.columns:
        data['INT_NPM1_pos_FLT3_neg'] = ((data['GENE_NPM1'] > 0) & (data['GENE_FLT3'] == 0)).astype(int)
        
    # TP53 + Complex Karyotype (Very Poor Prognosis)
    if 'GENE_TP53' in data.columns and 'Complex_Karyotype' in data.columns:
        data['INT_TP53_Complex'] = ((data['GENE_TP53'] > 0) & (data['Complex_Karyotype'] > 0)).astype(int)

    return data

df = add_interactions(df)
df_eval = add_interactions(df_eval)

# Add interactions to feature list
new_interactions = ['INT_NPM1_pos_FLT3_neg', 'INT_TP53_Complex']
total_features.extend([f for f in new_interactions if f in df.columns])

# 2. Clinical Data Transformation
clinical_cols = ['BM_BLAST', 'WBC', 'ANC', 'PLT', 'HB', 'MONOCYTES']

# Log transform skewed numericals (log1p handles 0s)
for col in clinical_cols:
    df[col] = np.log1p(df[col])
    df_eval[col] = np.log1p(df_eval[col])

# Fill Missing Clinical Data
imputer = SimpleImputer(strategy='median')
df[clinical_cols] = imputer.fit_transform(df[clinical_cols])
df_eval[clinical_cols] = imputer.transform(df_eval[clinical_cols])

total_features.extend(clinical_cols)

# Final Feature Cleanup (Remove duplicates)
total_features = list(set(total_features))
# Ensure all features exist in both dfs
total_features = [f for f in total_features if f in df.columns and f in df_eval.columns]

print(f"Final Feature Count: {len(total_features)}")

Final Feature Count: 1771


## 7. Model training with scaling

In [150]:
# Cell 7: Train Model

# Prepare X and y
X = df[total_features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# Split (for local validation)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.18, random_state=42)

# Create Pipeline: Scale -> CoxNet
# l1_ratio=0.5 means 50% Lasso (Selection) and 50% Ridge (Regularization)
# alpha_min_ratio controls the grid of alphas checked
pipeline = make_pipeline(
    StandardScaler(),
    CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.1, max_iter=2000)
)

print("Fitting model...")
pipeline.fit(X_train, y_train)

# Evaluate
c_index_train = concordance_index_ipcw(y_train, y_train, pipeline.predict(X_train), tau=5)[0]
c_index_test = concordance_index_ipcw(y_train, y_test, pipeline.predict(X_test), tau=5)[0]

print(f"Train C-Index: {c_index_train:.4f}")
print(f"Test C-Index:  {c_index_test:.4f}")

Fitting model...
Train C-Index: 0.7573
Test C-Index:  0.6954


## 8. Generate submission

In [ ]:
# Cell 8: Submission

# Ensure X_eval has the exact same columns in same order
X_eval_final = df_eval[total_features]

# Predict risk scores
predictions = pipeline.predict(X_eval_final)

# Create Submission DataFrame
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': predictions
})

# Save
submission.to_csv("submission/submission_improved_cox.csv", index=False)
print("Submission saved to submission_improved_cox.csv")
submission.head()

Submission saved to submission_improved_cox.csv


,ID,risk_score
0,KYW1,1.069016
1,KYW2,0.716306
2,KYW3,-0.209321
3,KYW4,0.691839
4,KYW5,0.269588
